In [ ]:

import resource
resource.setrlimit(resource.RLIMIT_STACK, (resource.RLIM_INFINITY, resource.RLIM_INFINITY)) 

In [ ]:
import os
import re
from pathlib import Path
from typing import Optional, Tuple
import subprocess


def resolve_molden_path(molden_folder_path: str, filename: str) -> Path:
    
    folder = Path(molden_folder_path).expanduser().resolve()
    
    direct = Path(filename)
    if direct.is_file():
        return direct

    base = os.path.splitext(os.path.basename(filename))[0]
    candidates = [
        folder / f"{base}.molden.input",
        folder / f"{base}.molden",
        folder / f"{base}.MOLDEN",
        folder / f"{base}.Molden",
        folder / f"{base}.mld",
    ]
    for p in candidates:
        if p.is_file():
            return p

    
    for p in folder.glob(f"{base}*"):
        if p.is_file() and p.suffix.lower() in {".molden", ".input", ".mld"}:
            return p

    raise FileNotFoundError("Public message removed for release.")


def get_molden_charge(molden_path: os.PathLike) -> Optional[int]:
    
    p = Path(molden_path)
    if not p.is_file():
        return None

    try:
        with open(p, "r", encoding="utf-8", errors="ignore") as fh:
            lines = fh.readlines()
    except Exception:
        return None

    
    sum_Z = 0
    in_atoms = False
    for i, line in enumerate(lines):
        s = line.strip()
        if not s:
            continue
        if s.startswith('[') and s.upper().startswith('[ATOMS'):
            in_atoms = True
            continue
        if in_atoms:
            if s.startswith('['):  
                in_atoms = False
                continue
            
            toks = s.split()
            if len(toks) >= 3:
                
                z_token = toks[2]
                if re.fullmatch(r"\d+", z_token):
                    sum_Z += int(z_token)
                    continue
            
            ints = [int(t) for t in toks if re.fullmatch(r"-?\d+", t)]
            if len(ints) >= 2:
                sum_Z += ints[1]
            elif len(ints) >= 1:
                
                sum_Z += ints[0]

    
    nelec = 0.0
    in_mo = False
    occup_re = re.compile(r"Occup\s*=\s*([\-+]?\d+(?:\.\d+)?(?:[EeDd][\-+]?\d+)?)", re.IGNORECASE)
    for line in lines:
        s = line.strip()
        if s.startswith('[') and s.upper().startswith('[MO]'):
            in_mo = True
            continue
        if in_mo:
            if s.startswith('['):   
                in_mo = True if s.upper().startswith('[MO]') else False
            m = occup_re.search(s)
            if m:
                occ = float(m.group(1).replace('D', 'E'))
                nelec += occ

    if sum_Z == 0 or nelec == 0.0:
        return None

    
    charge = int(round(sum_Z - nelec))
    return charge


def calculate_minmax_ESP(molden_folder_path: str, filename: str) -> Tuple[float, float, float, float]:
    
    folder = Path(molden_folder_path).expanduser().resolve()
    molden_file = resolve_molden_path(folder, filename)
    if not molden_file.is_file():
        raise FileNotFoundError("Public message removed for release.")

    cmd = ["Multiwfn", str(molden_file)]
    seq = "\n".join(["12", "0", "-1", "-1", "q"]) + "\n"

    proc = subprocess.run(
        cmd, input=seq, text=True, cwd=folder,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, check=False
    )
    out = proc.stdout or ""

    ESP_min_au: Optional[float] = None
    ESP_min_eV: Optional[float] = None
    ESP_max_au: Optional[float] = None
    ESP_max_eV: Optional[float] = None

    section = None  
    star_line_re = re.compile(
        r"^\*\s*\d+\s+([\-+]?\d+(?:\.\d+)?(?:[EeDd][\-+]?\d+)?)\s+([\-+]?\d+(?:\.\d+)?(?:[EeDd][\-+]?\d+)?)"
    )

    for raw in out.splitlines():
        line = raw.rstrip("\n")
        if "The number of surface minima" in line:
            section = "min"; continue
        if "The number of surface maxima" in line:
            section = "max"; continue

        if section in ("min", "max") and line.lstrip().startswith("*"):
            m = star_line_re.match(line.lstrip())
            if m:
                au = float(m.group(1).replace("D", "E"))
                ev = float(m.group(2).replace("D", "E"))
                if section == "min":
                    ESP_min_au, ESP_min_eV = au, ev
                else:
                    ESP_max_au, ESP_max_eV = au, ev
                section = None  

    if None in (ESP_min_au, ESP_min_eV, ESP_max_au, ESP_max_eV):
        raise RuntimeError("Public message removed for release.")

    return ESP_min_au, ESP_min_eV, ESP_max_au, ESP_max_eV

In [ ]:
import os                       
import subprocess               
import signal                   
from pathlib import Path        
from typing import List         

def batch_generate_esp_cub(molden_folder_path: str) -> List[Path]:
    
    
    input_dir = Path(molden_folder_path).expanduser().resolve()

    
    if not input_dir.is_dir():
        raise NotADirectoryError("Public message removed for release.")

    
    cmd_sequence = "\n".join(["5", "1", "3", "2", "0", "5", "12", "1", "2", "0", "q"])

    
    filename_list: List[Path] = []

    
    for molden_path in input_dir.glob("*.molden"):
        
        filename = molden_path.stem
        print(filename)

        
        density_tmp = input_dir / "density.cub"
        esp_tmp = input_dir / "totesp.cub"

        
        if density_tmp.exists():
            density_tmp.unlink()
        if esp_tmp.exists():
            esp_tmp.unlink()

        
        cmd = ["Multiwfn", str(molden_path), "-ESPrhoiso", "0.001"]

        
        with subprocess.Popen(
            cmd,
            stdin=subprocess.PIPE,      
            stdout=subprocess.PIPE,     
            stderr=subprocess.PIPE,
            text=True,                  
            cwd=input_dir               
        ) as proc:
            
            proc.communicate(input=cmd_sequence)

        
        density_new = input_dir / f"{filename}_density.cub"
        esp_new = input_dir / f"{filename}_ESP.cub"

        
        if density_tmp.exists():
            density_tmp.rename(density_new)
        else:
            print("Public message removed for release.")

        if esp_tmp.exists():
            esp_tmp.rename(esp_new)
        else:
            print("Public message removed for release.")
        
        
        filename_list.append(filename)
    
    return filename_list

In [ ]:


def compute_esp_limits(molden_folder_path: str, filename: str) -> Tuple[float, float, float, float]:
    
    molden_file = resolve_molden_path(molden_folder_path, filename)
    colorlow_au, colorhigh_au = -0.03, 0.03
    colorlow_ev, colorhigh_ev = -0.8, 0.8  

    charge = get_molden_charge(molden_file)
    if charge is not None and charge != 0:
        try:
            min_au, min_ev, max_au, max_ev = calculate_minmax_ESP(molden_folder_path, filename)
            colorlow_au, colorhigh_au = sorted([min_au, max_au])
            colorlow_ev, colorhigh_ev = sorted([min_ev, max_ev])
            print(f"[INFO] {Path(filename).stem}: charge={charge}, "
                  f"ESP range (a.u.)=({colorlow_au:.6f}, {colorhigh_au:.6f}); "
                  f"(eV)=({colorlow_ev:.4f}, {colorhigh_ev:.4f})")
        except Exception as e:
            print("Public message removed for release.")
    return colorlow_au, colorhigh_au, colorlow_ev, colorhigh_ev

In [ ]:
import os  

def generate_esp_vmd(molden_folder_path, filename: str) -> None:
    
    
    
    
    basename = os.path.splitext(os.path.basename(filename))[0]  
    colorlow_au, colorhigh_au, colorlow_ev, colorhigh_ev = compute_esp_limits(
        molden_folder_path, basename
    )

    
    
    vmd_script = f"""#This script is used to draw ESP colored molecular vdW surface (rho=0.001)

color scale method BWR
color Display Background white
axes location Off
display depthcue off
display rendermode GLSL
light 2 on
light 3 on
material change transmode EdgyGlass 1.0
material change specular EdgyGlass 0.15
material change shininess EdgyGlass 0.95
material change opacity EdgyGlass 0.7
material change outlinewidth EdgyGlass 0.9
material change outline EdgyGlass 0.5

#The maximum number of systems to be loaded
set nsystem 1
#Lower and upper limit of color scale of ESP (a.u.)
set colorlow  {colorlow_au:.8f}
set colorhigh {colorhigh_au:.8f}
# eV as unit (for reference)
# set colorlow  {colorlow_ev:.6f}
# set colorhigh {colorhigh_ev:.6f}

for {{set i 1}} {{${{i}}<=$nsystem}} {{incr i}} {{
set id [expr $i-1]
mol new {molden_folder_path}/{basename}_density.cub
mol addfile {molden_folder_path}/{basename}_ESP.cub
mol modstyle 0 $id CPK 1.000000 0.300000 22.000000 22.000000
mol addrep $id
mol modstyle 1 $id Isosurface 0.001000 0 0 0 1 1
mol modmaterial 1 $id EdgyGlass
mol modcolor 1 $id Volume 1
mol scaleminmax $id 1 $colorlow $colorhigh
}}
"""

    
    output_name = f"{molden_folder_path}/{basename}_ESPiso.vmd"  

    
    with open(output_name, "w", encoding="utf-8") as f:  
        f.write(vmd_script)  
    
    print(f"VMD script saved to: {os.path.abspath(output_name)}")  

In [ ]:
def generate_colorbar_vmd(molden_folder_path: str, filename: str) -> None:
    
    basename = os.path.splitext(os.path.basename(filename))[0]
    colorlow_au, colorhigh_au, colorlow_ev, colorhigh_ev = compute_esp_limits(
        molden_folder_path, basename
    )

    vmd_script = 
    out = Path(molden_folder_path) / f"{basename}_COLORBAR.vmd"
    with open(out, "w", encoding="utf-8") as f:
        f.write(vmd_script)
    print(f"Color bar VMD script saved to: {os.path.abspath(out)}")

In [ ]:




import os               
import subprocess        
import tempfile          
import textwrap          
from shutil import which 

def render_esp_with_vmd(molden_folder_path, filename: str) -> None:
    
    
    vmd_cmd = which('vmd')                         
    if vmd_cmd is None:                           
        raise FileNotFoundError(
            "Public message removed for release."
        )
    
    
    
    vmd_script = f"{molden_folder_path}/{filename}_ESPiso.vmd"         
    output_img  = f"{molden_folder_path}/{filename}_ESP.tga"           
    
    if not os.path.isfile(vmd_script):             
        raise FileNotFoundError(
            "Public message removed for release."
        )
    
    
    
    tcl_commands = textwrap.dedent().strip()                                  
    
    with tempfile.NamedTemporaryFile(
        mode='w', suffix='.tcl', delete=False
    ) as tcl_file:                                
        tcl_file.write(tcl_commands)              
        temp_tcl_path = tcl_file.name             
    
    
    
    try:
        result = subprocess.run(
            [vmd_cmd, '-dispdev', 'text', '-e', temp_tcl_path],
            check=False,                          
            stdout=subprocess.PIPE,               
            stderr=subprocess.PIPE,               
            text=True                             
        )
    finally:
        os.remove(temp_tcl_path)                  
    
    
    if result.returncode != 0:                    
        
        err_msg = (
            "Public message removed for release."
            f"--- stdout ---\n{result.stdout}\n"
            f"--- stderr ---\n{result.stderr}"
        )
        raise RuntimeError(err_msg)
    else:
        print("Public message removed for release.")       


In [ ]:
def render_colorbar_with_vmd(molden_folder_path: str, filename: str) -> None:
    
    vmd_cmd = which('vmd')
    if vmd_cmd is None:
        raise FileNotFoundError("Public message removed for release.")

    vmd_script = f"{molden_folder_path}/{filename}_COLORBAR.vmd"
    output_img = f"{molden_folder_path}/{filename}_color_bar.tga"
    if not os.path.isfile(vmd_script):
        raise FileNotFoundError("Public message removed for release.")

    tcl_commands = textwrap.dedent(f"""
        source "{vmd_script}"
        render TachyonInternal "{output_img}"
        quit
    """).strip()

    with tempfile.NamedTemporaryFile(mode='w', suffix='.tcl', delete=False) as tcl_file:
        tcl_file.write(tcl_commands)
        temp_tcl_path = tcl_file.name

    try:
        result = subprocess.run([vmd_cmd, '-dispdev', 'text', '-e', temp_tcl_path],
                                check=False, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    finally:
        os.remove(temp_tcl_path)

    if result.returncode != 0:
        raise RuntimeError("Public message removed for release.")
    print("Public message removed for release.")

In [ ]:
filename_list = batch_generate_esp_cub(".")
for filename in filename_list:
    generate_esp_vmd(".", filename)
    render_esp_with_vmd(".", filename)
    generate_colorbar_vmd(".", filename)
    render_colorbar_with_vmd(".", filename)